# Pattern 3: notebook -> function -> GitHub API -> notebook

```
this notebook
   |  POST /api/GetRepoInfo  (parallel for each repo in list)
   v
Azure Function (function/function_app.py :: get_repo_info)
   |  GET https://api.github.com/repos/{owner}/{repo}
   v
GitHub REST API
   |  returns repo metadata
   v
function curates response, returns to notebook
   v
notebook collects results into a Spark DataFrame
```

The function is a thin proxy that adds:
- Auth (GitHub PAT via `GITHUB_TOKEN` app setting, raising rate limit 60/hr -> 5,000/hr)
- Field curation (returns 18 fields, not the full ~80 GitHub returns)
- Error normalization (404 / rate-limit / network errors all return the same JSON envelope)

Why proxy the API through a function instead of calling GitHub directly from the notebook?
- Hide the GitHub PAT (notebook never sees it)
- Throttle / cache centrally (e.g. add Redis later)
- Same pattern works for any API: SaaS apps, internal services, paid APIs

In [ ]:
# === Configure ===
function_url = "https://my-fn-app.azurewebsites.net/api/GetRepoInfo"
function_key = "REPLACE_WITH_FUNCTION_KEY"

repos = [
    ("Azure",     "azure-kusto-spark"),
    ("Azure",     "azure-sdk-for-python"),
    ("microsoft", "vscode"),
    ("microsoft", "TypeScript"),
    ("microsoft", "fabric-samples"),
    ("apache",    "spark"),
    ("delta-io",  "delta"),
    ("pandas-dev","pandas"),
    ("python",    "cpython"),
    ("not-a-real-org", "not-a-real-repo"),     # demonstrates 404 handling
]

concurrency      = 5     # parallel requests to the function
request_timeout  = 30    # per-request timeout (seconds)

print(f"Will fetch {len(repos)} repos with concurrency={concurrency}")

In [ ]:
# === Fan out: call function in parallel for each repo ===
import json, time, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

session = requests.Session()
session.headers.update({
    "Content-Type":    "application/json",
    "x-functions-key": function_key,
})

def _fetch_one(owner_repo):
    owner, repo = owner_repo
    t0 = time.time()
    try:
        r = session.post(function_url,
                         data=json.dumps({"owner": owner, "repo": repo}),
                         timeout=request_timeout)
        ms = (time.time() - t0) * 1000
        try:
            body = r.json()
        except ValueError:
            body = {"ok": False, "error": f"non-JSON response: {r.text[:200]}"}
        return {**body, "_http_status": r.status_code, "_latency_ms": round(ms, 1)}
    except requests.RequestException as e:
        return {"ok": False, "owner": owner, "repo": repo,
                "error": f"{type(e).__name__}: {e}",
                "_http_status": None, "_latency_ms": round((time.time()-t0)*1000, 1)}

results = []
wall_t0 = time.time()
with ThreadPoolExecutor(max_workers=concurrency) as ex:
    futures = {ex.submit(_fetch_one, r): r for r in repos}
    for fut in as_completed(futures):
        results.append(fut.result())
wall_s = time.time() - wall_t0

ok_count   = sum(1 for r in results if r.get("ok"))
fail_count = len(results) - ok_count
print(f"Fetched {len(results)} in {wall_s:.1f}s "
      f"({ok_count} ok, {fail_count} failed)")

In [ ]:
# === Build a Spark DataFrame from the results ===
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, ArrayType,
)

spark = SparkSession.builder.getOrCreate()

schema = StructType([
    StructField("ok",                   BooleanType()),
    StructField("owner",                StringType()),
    StructField("repo",                 StringType()),
    StructField("name",                 StringType()),
    StructField("description",          StringType()),
    StructField("stars",                IntegerType()),
    StructField("forks",                IntegerType()),
    StructField("watchers",             IntegerType()),
    StructField("open_issues",          IntegerType()),
    StructField("language",             StringType()),
    StructField("license",              StringType()),
    StructField("topics",               ArrayType(StringType())),
    StructField("updated_at",           StringType()),
    StructField("pushed_at",            StringType()),
    StructField("html_url",             StringType()),
    StructField("rate_limit_remaining", StringType()),
    StructField("instance_id",          StringType()),
    StructField("error",                StringType()),
    StructField("_http_status",         IntegerType()),
    StructField("_latency_ms",          StringType()),
])

# Only keep fields the schema defines, in the right order
def _normalize(r):
    return Row(**{f.name: r.get(f.name) for f in schema.fields})

df = spark.createDataFrame([_normalize(r) for r in results], schema=schema)
df.cache()
print(f"DataFrame: {df.count()} rows, {len(df.columns)} columns")

In [ ]:
# === Display: successful repos, sorted by stars ===
(df.filter(F.col("ok") == True)
   .select("name", "language", "stars", "forks",
           "open_issues", "license", "updated_at", "_latency_ms")
   .orderBy(F.desc("stars"))
   .show(50, truncate=False))

In [ ]:
# === Display: errors ===
(df.filter(F.col("ok") == False)
   .select("owner", "repo", "_http_status", "error")
   .show(50, truncate=False))

In [ ]:
# === Summary stats across all successful fetches ===
(df.filter(F.col("ok") == True)
   .agg(
       F.count("*").alias("repos_fetched"),
       F.sum("stars").alias("total_stars"),
       F.sum("forks").alias("total_forks"),
       F.sum("open_issues").alias("total_open_issues"),
       F.countDistinct("language").alias("distinct_languages"),
       F.countDistinct("instance_id").alias("function_instances_hit"),
   )
   .show(truncate=False))

print("\nFunction instance distribution (proves auto-scale on consumption plan):")
(df.filter(F.col("ok") == True)
   .groupBy("instance_id")
   .count()
   .orderBy(F.desc("count"))
   .show(truncate=False))

## Adapt this pattern to any API

The function (`get_repo_info` in `function_app.py`) is ~50 lines and easy to fork. To call a different API:

1. Change `url = f"https://api.github.com/repos/{owner}/{repo}"` to your target
2. Adjust `headers` (auth scheme, content type)
3. Map the upstream response fields into the curated dict at the bottom
4. Update the parameters block at the top of this notebook
5. Update the Spark schema in cell 4 to match your curated fields

Common adaptations:
- **Internal REST API** behind Entra: replace `_GITHUB_TOKEN` with `DefaultAzureCredential` + managed-identity token
- **Paginated API** (returns 100 items at a time): make `get_repo_info` follow `Link: rel="next"` headers and accumulate
- **Rate-limited paid API**: add `time.sleep(retry_after)` on 429 with `Retry-After` header
- **API requiring complex auth (OAuth flow)**: store the refresh token in Key Vault, function exchanges it for an access token

## Cost notes

- Consumption plan: ~$0.20 per million invocations. 10 repos = $0.000002.
- Outbound bandwidth to GitHub is free up to 5GB/mo per region.
- GitHub PAT scope: only needs `public_repo` (read-only) for public repos.